#### Exercise 4: Building a Simple Retrieval System with LangChain

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

In [3]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames
from langchain.chains import RetrievalQA

# 1. Load a document about AI
document_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
documents_object = document_loader.load()

# 2. Split the document into chunks
text_splitter1 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_textscontainer = text_splitter1.split_documents(documents_object)

# 3. Set up the embedding model. (Use an embedding model to create vector representations.)
embed_params1 = {
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True}, #'True' means that we want to see the words too, with its embeddings.
}

embedding_model1 = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual", #'ibm/slate-125m-english-rtrvr-v2' was not working
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params1,
)

# 4. Create a vector store
vector_store = Chroma.from_documents(chunks_textscontainer, embedding_model1) #takes chunks and converts it into embeddings according to rules in 'embedding_model1'

# 5. Create a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3}) #'3' means the retriever fetches only the top 3 most semantically relevant chunks provided from all the relevant documents given by our Chroma database 

# 6. Define a function to search for relevant information
def search_documents(query, top_k=3):
    """Search for documents relevant to a query"""
    # Use the retriever to get relevant documents
    docs = retriever.get_relevant_documents(query)
    
    # Limit to top_k if specified
    return docs[:top_k]

# 7. Test with a few queries
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = search_documents(query)
    # Print the results
    print(f"Found {len(results)} relevant documents:")
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}: {doc.page_content[:150]}...")
        print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



Query: What is LangChain?
Found 3 relevant documents:

Result 1: everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what y...
Source: https://python.langchain.com/v0.2/docs/introduction/

Result 2: everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what y...
Source: https://python.langchain.com/v0.2/docs/introduction/

Result 3: See the Installation instructions and Quickstart guide to get started building your own agents and applications with LangChain.
Use LangSmith to trace...
Source: https://python.langchain.com/v0.2/docs/introduction/

Query: How do retrievers work?
Found 3 relevant documents:

Result 1: everything around the model loop: the prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what y...
Source: https://python.langchain.com/v0.2/

#### TO NOTE:
The code is running successfully without crashing, which is great..<br>
However, from a functional standpoint, it isn't working correctly yet.<br>
If we look closely at our output: every single query returns the exact same three snippets,
and the text snippet ("everything around the model loop...") doesn't actually answer specific questions like "How do retrievers work?"
or "Why is document splitting important?"<br>
Here are the two reasons why this is happening and how to fix them:<br>
1. **The Truncation Parameter Is Destroying Your Embeddings (Main Issue)**
In our configuration, we have this setting:<br>
-> EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3<br>
This tells Watsonx to throw away everything in our text chunks except for the first 3 tokens (words) before creating the embedding vector. <br>Because the chunks all come from the same page, many of them start with the same few words, making their mathematical vectors virtually identical.<br>
Additionally, our queries are also being truncated to 3 tokens, destroying their actual meaning.<br>
**The Fix**: Increase this limit or remove it entirely so the model reads the full context. IBM Granite supports up to 512 or 1024 tokens depending on the setup.<br>

2. **The Web Page Has Very Little Text**
The URL we are scraping (https://python.langchain.com/v0.2/docs/introduction/) is a high-level landing page.<br>
It doesn't actually contain deep paragraphs explaining how retrievers work or why splitting is important.<br>
Because the document is so sparse, Chroma is just returning the few available chunks it has over and over again.<br>

This is the new code I've got from Gemini, just storing it here:<br>
It is the updated script with the truncation fixed. It also adds a chunk_overlap adjustment and a fallback mock document mechanism so we can actually test the queries properly if the webpage text remains too short:<br>

In [4]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

# 1. Load the document (or use fallback text for testing local retrieval)
document_loader2 = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
documents_object2 = document_loader2.load()

# 2. Split the document into chunks
text_splitter2 = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks2 = text_splitter2.split_documents(documents_object2)

# 3. Set up the embedding model with proper truncation limits
embed_params2 = {
    # Changed from 3 to 512 so the model actually reads your sentences!
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 512, 
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

embedding_model2 = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params2,
)

# 4. Create a vector store
# Added a unique collection name to prevent mixing old data in the cache
vector_store2 = Chroma.from_documents(
    chunks2, 
    embedding_model2,
    collection_name="langchain_intro_collection" 
)

# 5. Create a retriever
retriever2 = vector_store2.as_retriever(search_kwargs={"k": 2}) 

# 6. Define the search function
def search_documents(query2, top_k=2):
    # Fixed the typo here from 'earch_documents'
    docs2 = retriever2.invoke(query2) 
    return docs2[:top_k]

# 7. Test
test_queries2 = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query2 in test_queries2:
    print(f"\nQuery: {query2}")
    results2 = search_documents(query2)
    print(f"Found {len(results2)} relevant documents:")
    for i, doc in enumerate(results2):
        print(f"\nResult {i+1}: {doc.page_content[:150]}...")
        print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



Query: What is LangChain?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Found 2 relevant documents:

Result 1: LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available p...
Source: https://python.langchain.com/v0.2/docs/introduction/

Result 2: LangChain vs. LangGraph vs. Deep AgentsStart with Deep Agents for a “batteries-included” agent with features like automatic context compression, a vir...
Source: https://python.langchain.com/v0.2/docs/introduction/

Query: How do retrievers work?
Found 2 relevant documents:

Result 1: Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDeploymentObser...
Source: https://python.langchain.com/v0.2/docs/introduction/

Result 2: agent = create_agent(
    model="anthropic.claude-3-5-sonnet-20240620-v1:0",
    model_provider="bedrock_converse",
    tools=[get_weather],
    syste...
Source: https://python.langchain.com/v0.2/docs/introduction/

Query: 

### Phase 2: Diagnosing the Output & The "Garbage In, Garbage Out" Problem:

#### 1. What was wrong earlier and why?
In our very first attempt, the pipeline executed but suffered from two flaws:
* **The Error:** The code crashed because the `ibm/slate-125m-english-rtrvr-v2` embedding model was no longer supported.
* **The Duplicate Output:** Once we updated the model, every query returned the exact same three generic text fragments. This happened because `EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS` was set to `3`. This forced the embedding model to slice both the document chunks and the search queries down to just the first **three words**, completely destroying their semantic meaning.

#### 2. What changed in the new code and what improved?
* **Model Upgrade:** We switched to the supported `ibm/granite-embedding-278m-multilingual` model.
* **Full Context Retention:** We increased `TRUNCATE_INPUT_TOKENS` to `512`.
* **The Result:** The embedding model can now read entire sentences. The pipeline now successfully generates unique mathematical vectors, allowing it to differentiate between questions and retrieve distinct snippets for each query.

#### 3. The New Obstacle: "Garbage In, Garbage Out"
While our code architecture is now flawless, the retrieved snippets look chaotic (e.g., fetching strings of text like `componentsAgentsModelsMessagesTools...` or `Context Protocol (MCP)...`, or `LangChain overview - Docs...`). We have hit a classic data engineering problem: **Garbage In, Garbage Out (GIGO)**.

I first thought: *Isn't it a good thing that the pipeline doesn't answer questions missing from the document? If it made them up, wouldn't it be hallucinating?*

Yes, I was correct about the **generation (LLM)** stage, but we wanted an LLM to admit it doesn't know the answer. However, the problem here is happening at the **retrieval** stage.

The URL we scraped (`https://python.langchain.com/v0.2/docs/introduction/`) is a high-level website landing page. Instead of deep, explanatory paragraphs about RAG concepts, it is stuffed with website navigation menus, sidebars, and UI buttons. Because the page lacks actual answers to technical questions like *"Why is document splitting important?"*, the retriever is mathematically forced to pull the "closest matching" vectors it can find—which turn out to be fragments of the website's user interface.

If we passed these junk fragments to an LLM, the LLM would get confused(Forced Hallucination) by the irrelevant text and either generate a messy, inaccurate answer or waste expensive API tokens processing useless data.

---

### The Solution: Controlled Testing with Clean Data
To prove that our RAG pipeline functions perfectly when fed high-quality data, we will temporarily bypass the web scraper. We will manually feed the vector store clean, intentional documents that contain explicit answers to our questions.

Let's try this code, which will help us see true semantic search in action:

---

In [7]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

# 1. Provide clean, intentional test data
mock_docs = [
    Document(
        page_content="LangChain is a popular framework designed to simplify the creation of applications using large language models (LLMs). It provides tools to chain together prompts, models, and data sources.",
        metadata={"source": "local_docs"}
    ),
    Document(
        page_content="Retrievers are interfaces that return documents given an unstructured query. They use vector similarity search to find text chunks whose embeddings match the mathematical representation of the user's question.",
        metadata={"source": "local_docs"}
    ),
    Document(
        page_content="Document splitting is critical because LLMs have context window limits. Breaking large PDFs or web pages into smaller chunks ensures the model only processes relevant data without getting overwhelmed.",
        metadata={"source": "local_docs"}
    )
]

# 2. Embedding Setup (Keeping our fixed 512 token limit)
embed_params3 = {
    EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 512, 
    EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}

embedding_model3 = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params3,
)

# 3. Create a clean vector store instance
vector_store3 = Chroma.from_documents(
    mock_docs, 
    embedding_model3,
    collection_name="langchain_perfect_data_collection" 
)

retriever3 = vector_store3.as_retriever(search_kwargs={"k": 1}) 

# 4. Define Search
def search_clean_documents(query3):
    return retriever3.invoke(query3)

# 5. Test
test_queries3 = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query3 in test_queries3:
    print(f"\nQuery: {query3}")
    results3 = search_clean_documents(query3)
    for doc in results3:
        print(f"Matched Content: {doc.page_content}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



Query: What is LangChain?
Matched Content: LangChain is a popular framework designed to simplify the creation of applications using large language models (LLMs). It provides tools to chain together prompts, models, and data sources.

Query: How do retrievers work?
Matched Content: Retrievers are interfaces that return documents given an unstructured query. They use vector similarity search to find text chunks whose embeddings match the mathematical representation of the user's question.

Query: Why is document splitting important?
Matched Content: Document splitting is critical because LLMs have context window limits. Breaking large PDFs or web pages into smaller chunks ensures the model only processes relevant data without getting overwhelmed.


Now, if we look at this output, Every single query matched its exact corresponding answer flawlessly.

By feeding the system clean, intentional data, we eliminated the "Garbage In, Garbage Out" noise. The retriever didn't get confused by website menus because we gave it actual, relevant paragraphs to work with.

This new output shows True Semantic Retrieval Success. It proves:
By swapping out the messy webpage text with clean, structured documents, the output changed instantly.

Flawless Semantic Alignment: The query "How do retrievers work?" fetched only the chunk explaining retrievers. The query "Why is document splitting important?" fetched only the chunk explaining context window limits.

Vector Math is Working: This proves that the ibm/granite-embedding-278m-multilingual model and the Chroma vector database are working together exactly as intended. The system successfully converted the text into high-quality mathematical vectors and accurately measured the semantic similarity between the user's questions and the documents.

## Takeaway for Building Production RAG Pipelines:
This exercise highlights the most critical rule of building Retrieval-Augmented Generation (RAG) systems: The code is only as good as the data source. When scraping web pages or loading PDFs for a real application, we cannot just dump raw HTML or unformatted text into a vector store. We must use Data Cleansing techniques (like stripping out navigation bars, headers, footers, and advertisements) before splitting the text. When we feed our pipeline clean data, the retriever will fetch accurate context, preventing the LLM from hallucinating or generating confused answers.